In [1]:
from pyspark.sql import SparkSession

spark_conf = {
    # "spark.jars.packages": "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,software.amazon.awssdk:bundle:2.20.18,software.amazon.awssdk:url-connection-client:2.20.18,org.postgresql:postgresql:42.1.1",
    "spark.jars.packages": "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.iceberg:iceberg-aws-bundle:1.9.0,org.postgresql:postgresql:42.1.1",
    "spark.sql.catalog.spark_catalog": "org.apache.iceberg.spark.SparkSessionCatalog",
    "spark.sql.catalog.spark_catalog.types": "hive",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.catalog.jdbc_catalog.catalog-impl": "org.apache.iceberg.jdbc.JdbcCatalog",
    "spark.sql.catalog.jdbc_catalog": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.jdbc_catalog.warehouse": "s3a://spark-warehouse/jdbc_catalog",
    "spark.sql.catalog.jdbc_catalog.uri": "jdbc:postgresql://iceberg-postgres:5432/metastore_db",
    "spark.sql.catalog.jdbc_catalog.jdbc.user": "postgres",
    "spark.sql.catalog.jdbc_catalog.jdbc.password": "postgres",
    "spark.sql.catalog.jdbc_catalog.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
    "spark.sql.catalog.jdbc_catalog.s3.endpoint": "http://minio:9000",
    "spark.sql.catalog.jdbc_catalog.s3.path-style-access": True,
    "spark.sql.catalog.jdbc_catalog.s3.access-key-id": "minioadmin",
    "spark.sql.catalog.jdbc_catalog.s3.secret-access-key": "minioadmin",
    "spark.driver.extraJavaOptions": "-Daws.region=us-east-1",
    "spark.executor.extraJavaOptions": "-Daws.region=us-east-1",
}

spark = SparkSession \
    .builder \
    .appName("spark-nb") \
    .master("spark://spark-master:7077") \
    .config(map=spark_conf) \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
df = spark.read.format("parquet").load("s3a://raw-data/")

In [3]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [4]:
df = df.select("VendorID")

In [5]:
spark.sql("DROP TABLE IF EXISTS local_db.sample_iceberg_table_one_column PURGE")

DataFrame[]

In [6]:
df.writeTo("local_db.sample_iceberg_table_one_column").using("iceberg").create()

In [7]:
print(spark.sql("SHOW CREATE TABLE local_db.sample_iceberg_table_one_column").collect()[0].createtab_stmt)

CREATE TABLE spark_catalog.local_db.sample_iceberg_table_one_column (
  VendorID INT)
USING iceberg
LOCATION 's3a://spark-warehouse/hive/local_db/sample_iceberg_table_one_column'
TBLPROPERTIES (
  'current-snapshot-id' = '5235418408850234500',
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd')



In [8]:
spark.sql("SELECT * FROM local_db.sample_iceberg_table_one_column LIMIT 5").toPandas()

,VendorID
0,2
1,2
2,2
3,1
4,2


In [9]:
spark.sql("DROP TABLE IF EXISTS jdbc_catalog.jdbc_db.sample_iceberg_table_one_column PURGE")

DataFrame[]

In [10]:
sql = """
CREATE TABLE jdbc_catalog.jdbc_db.sample_iceberg_table_one_column (
  VendorID INT
)
USING iceberg
LOCATION 's3a://spark-warehouse/jdbc/local_db/sample_iceberg_table_one_column'
TBLPROPERTIES (
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd')
"""
spark.sql(sql)

DataFrame[]

In [11]:
df.writeTo("jdbc_catalog.jdbc_db.sample_iceberg_table_one_column").using("iceberg").append()

In [12]:
spark.sql("SELECT count(*) FROM jdbc_catalog.jdbc_db.sample_iceberg_table_one_column").show()

+--------+
|count(1)|
+--------+
|  167585|
+--------+



In [13]:
spark.sql("SELECT count(*) FROM spark_catalog.local_db.sample_iceberg_table_one_column").show()

+--------+
|count(1)|
+--------+
|  167585|
+--------+



In [14]:
spark.sql("""
INSERT INTO spark_catalog.local_db.sample_iceberg_table_one_column
SELECT
    VendorID
FROM jdbc_catalog.jdbc_db.sample_iceberg_table_one_column
""").show()

++
||
++
++



In [15]:
spark.sql("""
INSERT INTO jdbc_catalog.jdbc_db.sample_iceberg_table_one_column
SELECT
    VendorID
FROM spark_catalog.local_db.sample_iceberg_table_one_column
""").show()

++
||
++
++



In [16]:
spark.sql("SELECT count(*) FROM jdbc_catalog.jdbc_db.sample_iceberg_table_one_column").show()

+--------+
|count(1)|
+--------+
|  502755|
+--------+



In [17]:
spark.sql("SELECT count(*) FROM spark_catalog.local_db.sample_iceberg_table_one_column").show()

+--------+
|count(1)|
+--------+
|  335170|
+--------+



In [18]:
spark.stop()